In [24]:
from inspect import FrameInfo
from google.colab import drive
drive.mount('/content/drive')

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np
import seaborn as sns

font_path = '/content/drive/MyDrive/kwukdt/data-analysis/data-pre-processing/NanumGothic.ttf'

fm.fontManager.addfont(font_path)
mpl.rc('font', family = 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
titanic = pd.read_csv('/content/drive/MyDrive/kwukdt/data-analysis/data-pre-processing/train.csv')
print(titanic.shape)
titanic.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [26]:
df = titanic.copy()
print("DataFrame 'titanic' copied to 'df'.")
print(f"Shape of df: {df.shape}")

DataFrame 'titanic' copied to 'df'.
Shape of df: (891, 12)


In [27]:
df['Age'] = df['Age'].fillna(age_median)
print(f"Missing values in 'Age' filled with median: {age_median}")

df['Embarked'] = df['Embarked'].fillna(embarked_mode)
print(f"Missing values in 'Embarked' filled with mode: {embarked_mode}")

fare_median = df['Fare'].median()
df['Fare'] = df['Fare'].fillna(fare_median)
print(f"Missing values in 'Fare' filled with median: {fare_median}")

print("\nMissing values after imputation:")
print(df.isnull().sum())

Missing values in 'Age' filled with median: 28.0
Missing values in 'Embarked' filled with mode: S
Missing values in 'Fare' filled with median: 14.4542

Missing values after imputation:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64


In [28]:
import numpy as np

Q1_fare = df['Fare'].quantile(0.25)
Q3_fare = df['Fare'].quantile(0.75)
IQR_fare = Q3_fare - Q1_fare

lower_bound_fare = Q1_fare - 1.5 * IQR_fare
upper_bound_fare = Q3_fare + 1.5 * IQR_fare

df['Fare'] = np.where(df['Fare'] < lower_bound_fare, lower_bound_fare,
                        np.where(df['Fare'] > upper_bound_fare, upper_bound_fare, df['Fare']))

print(f"'Fare' column outlier treatment complete.")
print(f"Q1 (Fare): {Q1_fare}, Q3 (Fare): {Q3_fare}, IQR (Fare): {IQR_fare}")
print(f"Lower Bound (Fare): {lower_bound_fare}, Upper Bound (Fare): {upper_bound_fare}\n")

# Outlier treatment for 'Age' column
Q1_age = df['Age'].quantile(0.25)
Q3_age = df['Age'].quantile(0.75)
IQR_age = Q3_age - Q1_age

lower_bound_age = Q1_age - 1.5 * IQR_age
upper_bound_age = Q3_age + 1.5 * IQR_age

df['Age'] = np.where(df['Age'] < lower_bound_age, lower_bound_age,
                       np.where(df['Age'] > upper_bound_age, upper_bound_age, df['Age']))

print(f"'Age' column outlier treatment complete.")
print(f"Q1 (Age): {Q1_age}, Q3 (Age): {Q3_age}, IQR (Age): {IQR_age}")
print(f"Lower Bound (Age): {lower_bound_age}, Upper Bound (Age): {upper_bound_age}")

'Fare' column outlier treatment complete.
Q1 (Fare): 7.9104, Q3 (Fare): 31.0, IQR (Fare): 23.0896
Lower Bound (Fare): -26.724, Upper Bound (Fare): 65.6344

'Age' column outlier treatment complete.
Q1 (Age): 22.0, Q3 (Age): 35.0, IQR (Age): 13.0
Lower Bound (Age): 2.5, Upper Bound (Age): 54.5


In [29]:
df['Fare'] = np.log1p(df['Fare'])
print("'Fare' column log transformation complete.")

'Fare' column log transformation complete.


In [30]:
import re

# 1. Create 'Family_Size' column
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
print("'Family_Size' column created.")

# 2. Create 'Is_Alone' column
df['Is_Alone'] = 0
df.loc[df['Family_Size'] == 1, 'Is_Alone'] = 1
print("'Is_Alone' column created.")

# 3. Extract 'Title' from 'Name' column
df['Title'] = df['Name'].apply(lambda x: re.search(r'([A-Za-z]+)\.', x).group(1) if re.search(r'([A-Za-z]+)\.', x) else '')
print("'Title' column extracted.")

# 4. Simplify 'Title' categories
print(f"Original unique titles: {df['Title'].unique()}")

df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

print(f"Simplified unique titles: {df['Title'].unique()}")
print("Title categories simplified.")

'Family_Size' column created.
'Is_Alone' column created.
'Title' column extracted.
Original unique titles: ['Mr' 'Mrs' 'Miss' 'Master' 'Don' 'Rev' 'Dr' 'Mme' 'Ms' 'Major' 'Lady'
 'Sir' 'Mlle' 'Col' 'Capt' 'Countess' 'Jonkheer']
Simplified unique titles: ['Mr' 'Mrs' 'Miss' 'Master' 'Rare']
Title categories simplified.


In [31]:
columns_to_encode = ['Sex', 'Embarked', 'Pclass', 'Title']
df_encoded = pd.get_dummies(df[columns_to_encode], drop_first=False)

# Concatenate the original DataFrame with the new dummy variables
df = pd.concat([df, df_encoded], axis=1)

# Drop the original categorical columns
df.drop(columns=columns_to_encode, axis=1, inplace=True)

print("One-hot encoding complete for 'Sex', 'Embarked', 'Pclass', 'Title'.")
print(f"New shape of df: {df.shape}")
print("First 5 rows of updated DataFrame:")
print(df.head())

One-hot encoding complete for 'Sex', 'Embarked', 'Pclass', 'Title'.
New shape of df: (891, 21)
First 5 rows of updated DataFrame:
   PassengerId  Survived                                               Name  \
0            1         0                            Braund, Mr. Owen Harris   
1            2         1  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2            3         1                             Heikkinen, Miss. Laina   
3            4         1       Futrelle, Mrs. Jacques Heath (Lily May Peel)   
4            5         0                           Allen, Mr. William Henry   

    Age  SibSp  Parch            Ticket      Fare Cabin  Family_Size  ...  \
0  22.0      1      0         A/5 21171  2.110213   NaN            2  ...   
1  38.0      1      0          PC 17599  4.199221   C85            2  ...   
2  26.0      0      0  STON/O2. 3101282  2.188856   NaN            1  ...   
3  35.0      1      0            113803  3.990834  C123            2  ...   
4  35.0   

In [32]:
from sklearn.preprocessing import StandardScaler
print("StandardScaler imported.")

StandardScaler imported.


In [33]:
numerical_cols = df.select_dtypes(include=np.number).columns

# Exclude non-feature columns from scaling
non_feature_cols = ['PassengerId', 'SibSp', 'Parch', 'Family_Size', 'Is_Alone', 'Survived']

# Filter out columns that are not relevant for scaling, but ensure they exist in numerical_cols before removing
columns_to_scale = [col for col in numerical_cols if col not in non_feature_cols]

# Initialize StandardScaler
scaler = StandardScaler()

# Scale the selected numerical columns
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

print(f"Scaled numerical columns: {columns_to_scale}")
print("DataFrame with scaled numerical features:")
print(df[columns_to_scale].head())

Scaled numerical columns: ['Age', 'Fare']
DataFrame with scaled numerical features:
        Age      Fare
0 -0.583432 -0.937738
1  0.742685  1.563065
2 -0.251903 -0.843593
3  0.494038  1.313600
4  0.494038 -0.826943
